# CashFlow and CashFlowSchedule Patterns

CashFlow objects represent individual monetary transactions (contributions or payments), while CashFlowSchedule objects organize multiple cash flows into a timeline. This notebook explores creating, organizing, and querying cash flows.

**Topics covered:**
- Creating individual cash flows (contributions and payments)
- Building cash flow schedules
- Querying schedules by date ranges
- Aggregating and analyzing cash flow patterns
- Working with inflows (contributions) and outflows (payments)


## Import Required Modules

Let's start by importing the necessary classes:


In [ ]:
from datetime import date
from decimal import Decimal
from sinkingfund import CashFlow, CashFlowSchedule


## Creating Individual Cash Flows

A `CashFlow` represents a single monetary transaction. Cash flows can be:
- **Positive amounts**: Inflows (contributions, savings) - money coming in
- **Negative amounts**: Outflows (payments, withdrawals) - money going out

Each cash flow is linked to a `bill_id` and has a specific date.


In [ ]:
# Create contribution cash flows (positive amounts = money coming in).
contribution1 = CashFlow(
    bill_id="car_insurance",
    date=date(2025, 1, 15),
    amount=Decimal("150.00")  # Positive = contribution
)

contribution2 = CashFlow(
    bill_id="car_insurance",
    date=date(2025, 2, 15),
    amount=Decimal("150.00")
)

# Create a payment cash flow (negative amount = money going out).
payment = CashFlow(
    bill_id="car_insurance",
    date=date(2025, 6, 15),
    amount=Decimal("-900.00")  # Negative = payment
)

print("=== Cash Flow Objects ===")
print(f"Contribution 1: {contribution1.date} - ${contribution1.amount} ({'inflow' if contribution1.is_inflow else 'outflow'})")
print(f"Contribution 2: {contribution2.date} - ${contribution2.amount} ({'inflow' if contribution2.is_inflow else 'outflow'})")
print(f"Payment: {payment.date} - ${payment.amount} ({'inflow' if payment.is_inflow else 'outflow'})")


### Cash Flow Properties

CashFlow objects have helpful properties to identify the direction of money flow:


In [ ]:
# Check if cash flows are inflows or outflows.
contrib = CashFlow(bill_id="test", date=date(2025, 1, 1), amount=Decimal("100.00"))
pay = CashFlow(bill_id="test", date=date(2025, 1, 1), amount=Decimal("-100.00"))

print(f"Contribution: ${contrib.amount}")
print(f"  Is inflow? {contrib.is_inflow}")
print(f"  Is outflow? {contrib.is_outflow}")

print(f"\nPayment: ${pay.amount}")
print(f"  Is inflow? {pay.is_inflow}")
print(f"  Is outflow? {pay.is_outflow}")


## Building Cash Flow Schedules

A `CashFlowSchedule` is a collection of cash flows that represents a complete timeline of transactions. Schedules automatically maintain chronological ordering and provide query methods for analysis.


In [ ]:
# Create a schedule.
schedule = CashFlowSchedule()

# Add individual cash flows.
schedule.add_cash_flows(CashFlow(bill_id="example", date=date(2025, 1, 15), amount=Decimal("100.00")))
schedule.add_cash_flows(CashFlow(bill_id="example", date=date(2025, 2, 15), amount=Decimal("100.00")))
schedule.add_cash_flows(CashFlow(bill_id="example", date=date(2025, 3, 15), amount=Decimal("100.00")))

# Or add multiple at once.
contributions = [
    CashFlow(bill_id="example", date=date(2025, 4, 15), amount=Decimal("100.00")),
    CashFlow(bill_id="example", date=date(2025, 5, 15), amount=Decimal("100.00")),
]
schedule.add_cash_flows(contributions)

print(f"Schedule contains {len(schedule.cash_flows)} cash flows")
print("\nCash flows (automatically sorted by date):")
for flow in schedule.cash_flows:
    print(f"  {flow.date}: ${flow.amount}")


## Querying Schedules by Date Ranges

The `cash_flows_in_range()` method lets you filter cash flows within a specific date range. This is useful for analyzing periods (quarters, months, etc.).


In [ ]:
# Create a more complex schedule with contributions and a payment.
complex_schedule = CashFlowSchedule()

# Add monthly contributions for first half of year.
for month in range(1, 7):
    complex_schedule.add_cash_flows(
        CashFlow(
            bill_id="insurance",
            date=date(2025, month, 15),
            amount=Decimal("150.00")
        )
    )

# Add a payment in June.
complex_schedule.add_cash_flows(
    CashFlow(
        bill_id="insurance",
        date=date(2025, 6, 30),
        amount=Decimal("-900.00")
    )
)

print("=== Full Schedule ===")
for flow in complex_schedule.cash_flows:
    print(f"  {flow.date}: ${flow.amount}")

# Get Q1 cash flows (January - March).
q1_flows = complex_schedule.cash_flows_in_range(
    start_date=date(2025, 1, 1),
    end_date=date(2025, 3, 31)
)

print("\n=== Q1 Cash Flows (Jan-Mar) ===")
for flow in q1_flows:
    print(f"  {flow.date}: ${flow.amount}")


## Aggregating Cash Flows

Schedules provide several methods for aggregating cash flows:

1. **`total_amount_as_of_date()`**: Sum of all cash flows up to and including a specific date
2. **`total_amount_in_range()`**: Sum of all cash flows within a date range
3. **Filtering by type**: Exclude contributions or payouts using the `exclude` parameter


In [ ]:
# Calculate total up to a specific date.
total_jan = complex_schedule.total_amount_as_of_date(date(2025, 1, 31))
total_march = complex_schedule.total_amount_as_of_date(date(2025, 3, 31))
total_june = complex_schedule.total_amount_as_of_date(date(2025, 6, 30))

print("=== Cumulative Totals ===")
print(f"Total as of Jan 31: ${total_jan}")
print(f"Total as of Mar 31: ${total_march}")
print(f"Total as of Jun 30: ${total_june}")

# Calculate total in a date range.
q1_total = complex_schedule.total_amount_in_range(
    start_date=date(2025, 1, 1),
    end_date=date(2025, 3, 31)
)
print(f"\nQ1 Total (Jan-Mar): ${q1_total}")


### Filtering by Type (Contributions vs Payments)

You can filter cash flows to exclude contributions or payouts:


In [ ]:
# Total including everything.
total_all = complex_schedule.total_amount_as_of_date(date(2025, 6, 30))
print(f"Total (all flows): ${total_all}")

# Total excluding payouts (only contributions).
contributions_only = complex_schedule.total_amount_as_of_date(
    date(2025, 6, 30),
    exclude='payouts'  # Exclude negative amounts
)
print(f"Total (contributions only): ${contributions_only}")

# Total excluding contributions (only payouts).
payouts_only = complex_schedule.total_amount_as_of_date(
    date(2025, 6, 30),
    exclude='contributions'  # Exclude positive amounts
)
print(f"Total (payouts only): ${payouts_only}")

# Same filtering works with date ranges.
q1_contribs = complex_schedule.total_amount_in_range(
    start_date=date(2025, 1, 1),
    end_date=date(2025, 3, 31),
    exclude='payouts'
)
print(f"\nQ1 contributions only: ${q1_contribs}")


## Analyzing Cash Flow Patterns

Let's build a realistic example to analyze cash flow patterns:


In [ ]:
# Create a schedule for property tax savings.
property_tax_schedule = CashFlowSchedule()

# Bi-weekly contributions from January to October.
contrib_dates = [
    date(2025, 1, 15), date(2025, 1, 29),
    date(2025, 2, 12), date(2025, 2, 26),
    date(2025, 3, 12), date(2025, 3, 26),
    date(2025, 4, 9), date(2025, 4, 23),
    date(2025, 5, 7), date(2025, 5, 21),
    date(2025, 6, 4), date(2025, 6, 18),
    date(2025, 7, 2), date(2025, 7, 16), date(2025, 7, 30),
    date(2025, 8, 13), date(2025, 8, 27),
    date(2025, 9, 10), date(2025, 9, 24),
    date(2025, 10, 8), date(2025, 10, 22),
]

for contrib_date in contrib_dates:
    property_tax_schedule.add_cash_flows(
        CashFlow(bill_id="prop_tax", date=contrib_date, amount=Decimal("180.00"))
    )

# Payment in November.
property_tax_schedule.add_cash_flows(
    CashFlow(bill_id="prop_tax", date=date(2025, 11, 1), amount=Decimal("-3600.00"))
)

print("=== Property Tax Savings Schedule ===")
print(f"Total contributions: {len([f for f in property_tax_schedule.cash_flows if f.is_inflow])}")
print(f"Total contribution amount: ${property_tax_schedule.total_amount_as_of_date(date(2025, 10, 31), exclude='payouts')}")
print(f"Payment amount: ${abs(property_tax_schedule.cash_flows_in_range(date(2025, 11, 1), date(2025, 11, 1))[0].amount)}")

# Analyze by quarter.
print("\n=== Quarterly Analysis ===")
quarters = [
    ("Q1", date(2025, 1, 1), date(2025, 3, 31)),
    ("Q2", date(2025, 4, 1), date(2025, 6, 30)),
    ("Q3", date(2025, 7, 1), date(2025, 9, 30)),
    ("Q4", date(2025, 10, 1), date(2025, 12, 31)),
]

for quarter_name, start, end in quarters:
    q_flows = property_tax_schedule.cash_flows_in_range(start, end, exclude='payouts')
    q_total = property_tax_schedule.total_amount_in_range(start, end, exclude='payouts')
    print(f"{quarter_name}: ${q_total} ({len(q_flows)} contributions)")


### Getting Cash Flow Dates

The `cash_flow_dates_in_range()` method returns just the dates when cash flows occur:


In [ ]:
# Get all dates with cash flows in Q2.
q2_dates = property_tax_schedule.cash_flow_dates_in_range(
    start_date=date(2025, 4, 1),
    end_date=date(2025, 6, 30)
)

print("Q2 cash flow dates:")
for d in q2_dates:
    # Find the flow for this date to show amount.
    flows_on_date = [f for f in property_tax_schedule.cash_flows if f.date == d]
    for flow in flows_on_date:
        print(f"  {d}: ${flow.amount}")


## Summary

**Key Concepts:**

1. **CashFlow Objects**: Represent individual monetary transactions
   - Positive amounts = contributions (inflows)
   - Negative amounts = payments (outflows)
   - Linked to `bill_id` and have a specific `date`

2. **CashFlowSchedule Objects**: Collections of cash flows
   - Automatically maintain chronological ordering
   - Provide query methods for analysis

3. **Querying Methods**:
   - `cash_flows_in_range()`: Get flows within date range
   - `total_amount_as_of_date()`: Cumulative total up to a date
   - `total_amount_in_range()`: Total within a date range
   - `cash_flow_dates_in_range()`: Just the dates in a range

4. **Filtering**: Use `exclude='contributions'` or `exclude='payouts'` to filter by type

5. **Pattern Analysis**: Combine query methods to analyze cash flow patterns by time periods

**Next Steps:**
- See how schedules integrate with envelopes in the Envelope Lifecycle Management notebook
- Explore how schedulers generate contribution schedules automatically
- Learn about allocation strategies that distribute funds across multiple envelopes
